# 05 · Attention (Queries, Keys, Values)

**Goal:** build scaled dot-product attention step by step and see what each piece does.

Concepts: *query, key, value*, *scores*, *softmax*, *scaling by √d_k*, *causal mask*, *padding mask*, *batched attention*.

Standalone — needs `numpy` and `matplotlib`.

## Words you'll meet
- **Token**: one piece of a sentence (here, a word). Like one Lego brick in a sentence built from bricks.
- **Vector**: a short list of numbers describing something, like GPS coordinates describe a place.
- **Matrix**: a grid of numbers (rows and columns), like a school timetable or a class register.
- **Query / key / value**: the question you ask, the label on a library book, and the content inside the book.
- **Score**: how well a query matches a key; a higher number means a better match.
- **Softmax**: a recipe that turns any list of scores into shares that add up to 100%, like splitting your attention among options.
- **Mask**: a piece of card that covers part of an exam paper so you cannot peek at those answers.

## 1. Intuition: a soft dictionary lookup
A normal dictionary: given a **query**, find the matching **key**, return its **value**.
Attention does this *softly*: compare the query to **all** keys, get similarity scores, turn them into weights that sum to 1, and return the **weighted average of the values**.

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

> **Think of it like...** a library. The query is the question you ask, each key is the label on a book spine, and each value is what is inside the book. You compare your question with all labels, then read a blend of the books, giving more time to the better matches. Softmax is how you share 100% of your attention among the books.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)        # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

print(softmax(np.array([1.0, 2.0, 3.0])).round(3), "-> sums to 1")

> 🧒 **In plain words:** We built a helper called `softmax`. It takes a list of scores and turns them into shares that add up to 1 (100%).
> The line with `x.max` just subtracts the biggest number first so the maths never explodes; the shares come out the same.
> `np.exp` makes every number positive and makes bigger scores grow much faster.
> **Notice:** for scores 1, 2, 3 we get 0.09, 0.245 and 0.665. The biggest score gets most of the share, but the others still get something.
> **Try it:** change the list to `[1.0, 2.0, 10.0]` and expect the last share to be almost 1.

## 2. A hand-made lookup
Three keys (one-hot), three values. A query equal to a key should retrieve that key's value.

In [ ]:
keys   = np.eye(3) * 5                       # strong one-hot keys
values = np.array([[10., 0.], [0., 10.], [5., 5.]])

query = np.array([0., 5., 0.])               # "I'm looking for key #1"
scores = keys @ query                        # similarity with every key
weights = softmax(scores)
print("scores :", scores)
print("weights:", weights.round(3))
print("output :", (weights @ values).round(2), " ~ values[1] =", values[1])

> 🧒 **In plain words:** We made 3 library books (the keys) and the content of each one (the values). `np.eye(3) * 5` is a grid with 5s on the diagonal and 0s elsewhere, so each key shouts about exactly one topic.
> The query `[0, 5, 0]` says: I want book number 1 (counting from 0).
> `keys @ query` is a quick way to compare the query with every key at once: multiply matching numbers and add them up.
> **Notice:** scores are `[0, 25, 0]`, so softmax gives weights `[0, 1, 0]`: all attention on book 1. The output `[0, 10]` is exactly that book's content.
> Think of it like asking a librarian for one specific book and getting exactly that.

A query *between* two keys blends their values:

In [ ]:
query = np.array([2.5, 2.5, 0.])
w = softmax(keys @ query)
print("weights:", w.round(3), "  output:", (w @ values).round(2))

> 🧒 **In plain words:** This time the query points halfway between key 0 and key 1.
> Both keys match equally well, so softmax shares attention 50/50 and the third key gets 0.
> **Notice:** weights are `[0.5, 0.5, 0]` and the output is `[5, 5]`, the average of the first two values `[10, 0]` and `[0, 10]`.
> **Try it:** change the query to `[4, 1, 0]` and expect most of the weight on key 0.

## 3. Where Q, K, V come from
In a Transformer they are **learned linear projections of the same token vectors** `X` (shape `(T, d_model)`):

`Q = X W_q`, `K = X W_k`, `V = X W_v`

Each token asks a question (`Q`), advertises what it contains (`K`), and offers content (`V`).

> **Think of it like...** every pupil in the class writing three things about themselves from the same self-description: a question they want answered (query), a name badge showing what they are about (key) and a note they can share (value).

In [ ]:
rng = np.random.default_rng(0)
tokens = ["the", "cat", "sat", "down"]
T, d_model, d_k = len(tokens), 8, 4

X = rng.normal(size=(T, d_model))             # pretend these are embeddings + PE
W_q = rng.normal(size=(d_model, d_k))
W_k = rng.normal(size=(d_model, d_k))
W_v = rng.normal(size=(d_model, d_k))

Q, K, V = X @ W_q, X @ W_k, X @ W_v
print("X", X.shape, " Q", Q.shape, " K", K.shape, " V", V.shape)

> 🧒 **In plain words:** We invented 4 words and gave each a random list of 8 numbers (`X`; 4 rows, one per word, by 8 columns). Real models learn these numbers; here they are random.
> `W_q`, `W_k`, `W_v` are three grids of random numbers that turn each word's 8 numbers into 4 new numbers.
> `X @ W_q` means: send every word through the first grid. We do it three times to get the questions `Q`, labels `K` and contents `V`.
> **Notice:** the printout shows shapes. `X (4, 8)` means 4 rows by 8 columns. `Q`, `K`, `V` are each 4 rows by 4 columns: one short list per word.
> It is like each pupil writing a question, a name badge and a note, all based on the same self-description.

## 4. Scores → weights → output, step by step

In [ ]:
scores = Q @ K.T / np.sqrt(d_k)              # (T, T): token i's query vs token j's key
print("scores (T,T):"); print(scores.round(2))

weights = softmax(scores, axis=-1)           # each ROW sums to 1
print("\nweights row sums:", weights.sum(-1))

out = weights @ V                            # (T, T) @ (T, d_k) -> (T, d_k)
print("output:", out.shape)

> 🧒 **In plain words:** `Q @ K.T` compares every word's question with every word's label. `.T` flips the grid so rows become columns, which makes the multiplication line up. Dividing by `np.sqrt(d_k)` (here 2) keeps the numbers from getting too big.
> The result `scores` is a 4 by 4 table: row = the word asking, column = the word being looked at.
> `softmax(scores, axis=-1)` turns each row into shares, so every row adds up to 1. `weights @ V` then mixes the contents using those shares.
> **Notice:** the scores include big and negative numbers like 6.09 and -5.5; the row sums print as `[1. 1. 1. 1.]`; the output is 4 by 4 again.

In [ ]:
def show_attn(w, labels_q, labels_k, title="attention weights"):
    plt.figure(figsize=(4, 3.5))
    plt.imshow(w, cmap="Blues", vmin=0, vmax=1)
    plt.xticks(range(len(labels_k)), labels_k); plt.yticks(range(len(labels_q)), labels_q)
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            plt.text(j, i, f"{w[i,j]:.2f}", ha="center", va="center", fontsize=8)
    plt.xlabel("key (attended to)"); plt.ylabel("query (attending)"); plt.title(title); plt.show()

show_attn(weights, tokens, tokens)

> 🧒 **In plain words:** This defines a drawing helper and draws the weights table as a coloured grid. Darker blue means more attention.
> The two loops write the number in each square. Rows are the word asking, columns are the word being looked at.
> **Notice:** each row's numbers add up to 1. Some squares are dark (strong focus), others pale. For example the second row focuses on the last column, and the third row focuses mostly on the first.
> **Try it:** call `show_attn(weights, tokens, tokens, "my title")` to change the title.

Row `i` = how token `i` distributes its attention over all tokens. (Weights are random here because `W_q/W_k` are untrained; training shapes them.)

## 5. Why divide by √d_k?
If `q` and `k` have independent unit-variance entries, `q·k` has variance **`d_k`**. For large `d_k` the scores become huge, softmax saturates to nearly one-hot, and gradients vanish. Dividing by `√d_k` restores unit variance.

> **Think of it like...** a volume knob. With many numbers added up, the scores get very loud, and softmax then shouts about just one winner. Dividing by the square root of `d_k` turns the volume back down so several options can still be heard.

In [ ]:
rng = np.random.default_rng(0)
print(f"{'d_k':>5} {'std(q·k)':>10} {'std scaled':>11} {'max weight (unscaled)':>22} {'max weight (scaled)':>20}")
for d_k_ in [4, 64, 1024]:
    q = rng.normal(size=(2000, d_k_)); k = rng.normal(size=(2000, 10, d_k_))
    s = np.einsum("nd,nmd->nm", q, k)
    print(f"{d_k_:5d} {s.std():10.2f} {(s/np.sqrt(d_k_)).std():11.2f} "
          f"{softmax(s).max(-1).mean():22.2f} {softmax(s/np.sqrt(d_k_)).max(-1).mean():20.2f}")

> 🧒 **In plain words:** For each key length `d_k` (4, 64, 1024) we made 2000 random queries and 2000 sets of 10 random keys, then compared them. `np.einsum("nd,nmd->nm", q, k)` is a compact way of saying: for each example, multiply the query with each of its 10 keys and add up.
> The table shows how spread-out the scores are (`std`), and the biggest weight softmax gives, with and without dividing by the square root.
> **Notice:** without scaling, the spread grows with `d_k` (2.01, 7.93, 31.87) and the biggest weight climbs to 0.87 and 0.96, nearly all the attention on one key. With scaling the spread stays near 1 and the biggest weight stays around 0.3.
> **Try it:** add 4096 to the list of sizes and expect the unscaled max weight to get even closer to 1.

Without scaling and with large `d_k`, one key takes ~all the weight (max weight 0.87–0.96 vs ~0.3 when scaled) regardless of content.

## 6. Causal mask (for the decoder)
When generating, token `t` must not see tokens `> t` (they don't exist yet; at training time they'd be cheating). Set scores for future positions to `-inf` **before** softmax so their weight is exactly 0.

> **Think of it like...** covering the rest of an exam paper with a card so you can only read the questions up to the one you are on. Minus infinity is the ink that blacks out the covered part.

In [ ]:
causal = np.tril(np.ones((T, T), dtype=bool))     # True = allowed
print(causal.astype(int))

masked = np.where(causal, scores, -np.inf)
w_causal = softmax(masked)
show_attn(w_causal, tokens, tokens, "causal attention")

> 🧒 **In plain words:** `np.tril(np.ones((T, T)))` makes a 4 by 4 grid of ones, then keeps only the lower-left triangle (`tril` means triangle lower). 1 means you may look, 0 means covered.
> `np.where(causal, scores, -np.inf)` goes through every square: if allowed keep the score, otherwise write minus infinity. After softmax, minus infinity becomes exactly 0.
> **Notice:** the printed grid is a staircase of 1s. In the picture, everything above the diagonal is 0.00 and each row still adds up to 1. The first word can only look at itself, so it gets 1.00.

## 7. Padding mask
Same trick for `<PAD>` keys (notebook 02): nobody should attend to padding.

In [ ]:
key_is_real = np.array([True, True, True, False])      # last token is <PAD>
masked = np.where(key_is_real[None, :], scores, -np.inf)
w_pad = softmax(masked)
print(w_pad.round(2))
print("\nattention to PAD column:", w_pad[:, 3])

> 🧒 **In plain words:** The list `key_is_real` says which words are real and which is just filler (`<PAD>`, used to make sentences the same length). `key_is_real[None, :]` stretches this one row so it applies to every row of the scores table.
> Then the same `np.where` trick hides the filler column.
> **Notice:** the fourth column of the weights is all 0.00, and the other columns in each row still share 100%. The last print confirms `[0. 0. 0. 0.]`.
> It is like ignoring blank pages in a book.

## 8. Self vs. cross attention
- **Self-attention**: Q, K, V all come from the *same* sequence (encoder, decoder's first sub-layer).
- **Cross-attention**: Q from the **decoder**, K and V from the **encoder output** — lets each output token look at the input.

Shapes: `weights` is `(T_query, T_key)`, so the two lengths may differ.

In [ ]:
rng = np.random.default_rng(1)
T_enc, T_dec = 5, 3
Q = rng.normal(size=(T_dec, d_k))
K = rng.normal(size=(T_enc, d_k)); V = rng.normal(size=(T_enc, d_k))
w = softmax(Q @ K.T / np.sqrt(d_k))
print("weights", w.shape, "= (T_dec, T_enc);  output", (w @ V).shape, "= (T_dec, d_k)")

> 🧒 **In plain words:** Here the questions come from 3 output words (`T_dec`), while the keys and values come from 5 input words (`T_enc`), so the lengths differ.
> Everything else is the same recipe as before. `.shape` just reports rows and columns.
> **Notice:** the weights table is 3 rows by 5 columns (each of the 3 output words shares attention over the 5 input words) and the output is 3 rows by 4 columns: one new list per output word.
> Like a translator glancing back over the whole original sentence for each word they write.

## 9. Batched attention function
Put it all together for tensors with a leading batch dimension; `@` broadcasts over `B`.

In [ ]:
def attention(Q, K, V, mask=None):
    """Q:(B,Tq,dk) K:(B,Tk,dk) V:(B,Tk,dv) mask: bool broadcastable to (B,Tq,Tk), True=allowed"""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)     # (B,Tq,Tk)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    w = softmax(scores, axis=-1)
    return w @ V, w

B, T, d_k = 2, 4, 4
rng = np.random.default_rng(0)
Q = rng.normal(size=(B, T, d_k)); K = rng.normal(size=(B, T, d_k)); V = rng.normal(size=(B, T, d_k))

out, w = attention(Q, K, V, mask=np.tril(np.ones((T, T), bool)))
print("out", out.shape, " weights", w.shape)
print("upper triangle is zero:", np.allclose(np.triu(w[0], k=1), 0))

> 🧒 **In plain words:** We put all the steps into one reusable function `attention`. Now it can handle a whole batch: `B = 2` sentences at once, each with `T = 4` words.
> `K.transpose(0, 2, 1)` flips each sentence's key grid, keeping the sentence number in front (like `.T` but for a stack of grids).
> If a `mask` is given, forbidden squares become minus infinity before softmax.
> `np.triu(w[0], k=1)` picks the part above the diagonal of the first sentence's weights, and `np.allclose(..., 0)` checks it is all zero.
> **Notice:** both outputs have shape `(2, 4, 4)` (2 sentences, 4 words, 4 numbers each) and the check prints `True`, so no word peeked at the future.

## 10. Cost: attention is `O(T²)`
The `(T, T)` weight matrix grows quadratically with sequence length — the main reason long contexts are expensive.

In [ ]:
for T_ in [128, 1024, 8192, 100_000]:
    print(f"T={T_:>7,}  weight matrix entries per head per example = {T_*T_:>16,}")

> 🧒 **In plain words:** A loop that prints how many squares the weights table has for different sentence lengths. The table has T rows and T columns, so T times T squares.
> **Notice:** 128 words gives 16,384 squares, 1,024 gives about a million, and 100,000 gives 10,000,000,000. Ten times longer text means a hundred times more work.
> It is like everyone in a class greeting everyone else: doubling the class makes four times as many greetings.
> **Try it:** add 200 to the list and expect 40,000.

## Recap
1. `Q, K, V = X W_q, X W_k, X W_v`
2. `scores = Q Kᵀ / √d_k`  → `(T_q, T_k)`
3. mask (causal / padding) → softmax over keys → weights
4. `output = weights @ V`

➡ Next: **06 · Multi-Head Attention and the Encoder Block**.